In [0]:
#Create a DataFrame with a case-mismatch and a genuinely invalid value
from pyspark.sql.functions import col, upper

data = [(1, "Active"), (2, "active"), (3, "Inactive"), (4, "Suspended"), (5, "Pending")]
df = spark.createDataFrame(data, ["id", "status"])
df.display()

In [0]:
#Basic isin() check — find invalid rows
valid_statuses = ["Active", "Inactive", "Pending"]

invalid_rows = df.filter(~col("status").isin(valid_statuses))
invalid_rows.display()

In [0]:
#Normalize case and confirm the fix
valid_statuses_upper = [s.upper() for s in valid_statuses]
invalid_rows_ci = df.filter(~upper(col("status")).isin(valid_statuses_upper))
invalid_rows_ci.display()



In [0]:
#Find which unexpected values actually exist
unexpected_values = df.filter(~upper(col("status")).isin(valid_statuses_upper)).select("status").distinct()
unexpected_values.display()

In [0]:
# Measure and report the failure percentage
total = df.count()
failures = df.filter(~upper(col("status")).isin(valid_statuses_upper)).count()
print(f"Invalid status values: {failures} out of {total} ({failures/total*100:.1f}%)")


In [0]:
# Create orders DataFrame with region data (valid and invalid)
orders_data = [
    (1001, "2024-01-10", "APAC", 1500.00),
    (1002, "2024-01-11", "emea", 2300.00),      # lowercase - should be valid
    (1003, "2024-01-12", "Americas", 1800.00),  # mixed case - should be valid
    (1004, "2024-01-13", "LATAM", 950.00),      # invalid region
    (1005, "2024-01-14", "apac", 2100.00),      # lowercase - should be valid
    (1006, "2024-01-15", "NA", 1200.00),        # invalid region
    (1007, "2024-01-16", "EMEA", 3400.00),
    (1008, "2024-01-17", "MEA", 1700.00),       # invalid region
    (1009, "2024-01-18", "AMERICAS", 2900.00),
    (1010, "2024-01-19", "Asia", 1600.00)       # invalid region
]

orders_columns = ["order_id", "order_date", "region", "amount"]
orders = spark.createDataFrame(orders_data, orders_columns)
orders.display()

print(f"Total orders: {orders.count()}")

In [0]:
#Putting It All Together — A Realistic Example
from pyspark.sql.functions import col, upper

valid_regions = ["APAC", "EMEA", "AMERICAS"]

orders_checked = orders.withColumn(
    "region_valid", upper(col("region")).isin(valid_regions)
)

invalid_regions_found = orders_checked.filter(~col("region_valid")).select("region").distinct()

print("Distinct invalid region values found:")
invalid_regions_found.display()

total = orders_checked.count()
failures = orders_checked.filter(~col("region_valid")).count()
print(f"Failing rows: {failures} out of {total} ({failures/total*100:.1f}%)")
